# 02 - Time-based split and features

In this notebook I split the data by time into train, validation and test, and then prepare the features.

**Rule:** I split first. Everything I learn from the data (averages for filling gaps, categories, scaling) comes from **train only** and is then applied to validation and test. Otherwise information from the future leaks into training and the scores look better than they really are.

| Part | Job |
|---|---|
| Train | the model learns from it |
| Validation | I tune with it: model choice, threshold, early stopping |
| Test | final score, used once at the end |

## 1. Load and merge the data

Same as in notebook 01: the two train files, joined with a left join on `TransactionID` so every transaction is kept.

In [150]:
import pandas as pd

# load the two train files (path goes up one folder from notebooks/)
df = pd.read_csv('../data/raw/train_transaction.csv')
df1 = pd.read_csv('../data/raw/train_identity.csv')

# left join keeps every transaction; device columns are NaN where there is no identity row
df_merged = pd.merge(df, df1, on='TransactionID', how='left')

# expected: (590540, 434)
df_merged.shape

(590540, 434)

## 2. Time-based split

The data covers about 26 weeks and is already in time order. I want roughly 70% train, 15% validation and 15% test, with train the earliest part and test the latest.

**My plan:**

| Part | Weeks | Why |
|---|---|---|
| Train | 0-17 | the past, the model learns from it |
| Validation | 18-21 | the next weeks, for tuning |
| Test | 22-26 | the most recent weeks, final score once |

I split on whole weeks so no week is cut in half, and it is easy to explain: "I trained on the first 18 weeks".

### Week number

This notebook needs the `week` column again. `TransactionDT` is in seconds, one week is 86,400 x 7 seconds, and `//` drops the decimals.

In [151]:
# week number of each transaction (0 to 26)
df_merged['week'] = df_merged['TransactionDT'] // (86400 * 7)

# check: should be 0 and 26
print(df_merged['week'].min(), df_merged['week'].max())

0 26


C:\Users\HP\AppData\Local\Temp\ipykernel_9788\4177173830.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_merged['week'] = df_merged['TransactionDT'] // (86400 * 7)


### Masks for the three parts

Each mask asks a question about the week for every row (True/False). The parts must not overlap: week 21 belongs to validation only, so test starts at 22.

**Watch out:** when two conditions are joined with `&`, each one needs its own brackets. Without them Python runs `&` before `>=` and builds a wrong mask without any error.

In [152]:
# a mask is True/False for every row; each one picks the weeks for one part
# train: weeks 0-17 (the past)
train_mask = df_merged['week'] <= 17

# validation: weeks 18-21; two conditions, each in its own brackets, joined with & (and)
validation_mask = (df_merged['week'] >= 18) & (df_merged['week'] <= 21)

# test: weeks 22-26 (the most recent part)
test_mask = df_merged['week'] >= 22

### Check: does every row land in exactly one part?

In [153]:
# summing True/False counts the True values = number of rows in each part
print(train_mask.sum(), validation_mask.sum(), test_mask.sum())

431125 77090 82325


**Result:**

| Part | Rows | Share |
|---|---|---|
| Train | 431,125 | 73% |
| Validation | 77,090 | 13% |
| Test | 82,325 | 14% |
| **Total** | **590,540** | 100% |

The three parts add up to exactly 590,540, so every transaction is in one part only: no gaps and no overlaps.

### Cut the data into three tables and check the fraud rate in each

The masks only point at rows (True/False). Filtering `df_merged` with a mask makes a real, separate table:

- `train` (weeks 0-17): the model learns from this table only, and everything in preprocessing (fill values, category lists, scaling) is learned from it too.
- `val` (weeks 18-21): used to compare models and choose the threshold. The model never learns from it.
- `test` (weeks 22-26): kept aside and used once at the end for the final score.

Clear names matter: if the test table had a name like `train2`, it would be easy to train on it by mistake, which is leakage.

From the weekly chart in notebook 01, the early weeks had less fraud, so train may be a bit below 3.5%.

In [154]:
# filter df_merged with each mask (no quotes: the mask is a variable, not a column name)
train = df_merged[train_mask]
val = df_merged[validation_mask]
test = df_merged[test_mask]

In [155]:
# fraud rate (%) in each part: mean of the 0/1 isFraud column x 100
# (overall rate for comparison: 3.50%)
print(train['isFraud'].mean() * 100)
print(val['isFraud'].mean() * 100)
print(test['isFraud'].mean() * 100)

3.5140620469701367
3.4310546114930602
3.4837534163376858


**Result:**

| Part | Weeks | Rows | Fraud rate |
|---|---|---|---|
| Train | 0-17 | 431,125 | 3.51% |
| Validation | 18-21 | 77,090 | 3.43% |
| Test | 22-26 | 82,325 | 3.48% |
| *All data* | | *590,540* | *3.50%* |

All three parts have almost the same fraud rate as the whole dataset. So the model learns from, is tuned on and is tested on data with the same amount of fraud, and any difference between validation and test scores later will not come from different fraud rates.

I expected train to be lower, because weeks 0-3 had little fraud. But weeks 4-10 had more fraud than average, and both are inside train, so they balance out.

## 3. Clues and answer: X and y

A model needs the data in two parts:

- **X:** the clues the model looks at (amount, product, card, device...)
- **y:** the answer it tries to predict (`isFraud`)

If `isFraud` stayed in X, the model would just read the answer and score 100%, but it would be useless in real life, where nobody knows the answer in advance. That is the most extreme kind of leakage.

Some other columns are not real clues either and have to be removed from X: `TransactionID`, `TransactionDT` and `week`.

*Why each of these would be a mistake as a clue:*

In [156]:
# y = the answer the model has to predict
y_train = train['isFraud']

In [157]:
# X = the clues: everything except the answer, the ID and the time columns
X_train = train.drop(columns=['isFraud', 'TransactionID', 'TransactionDT', 'week'])

**Why the ID and time columns are removed:**

- `TransactionID` is just a receipt number. It grows over time, so the model could memorise "IDs around 3,100,000 had more fraud", but new payments get IDs it has never seen.
- `TransactionDT` and `week` are the time of the payment. Every future payment has a later time than anything in training, so a pattern about specific times does not carry over. Time is used for the split, not as a clue.

A good clue describes the payment itself (amount, product, card, device, email) and means the same thing for a payment next month.

Next: the same X and y for validation and test.

In [158]:
# same X and y for validation and test, with the same 4 columns removed
y_val = val['isFraud']
X_val = val.drop(columns=['isFraud', 'TransactionID', 'TransactionDT', 'week'])

y_test = test['isFraud']
X_test = test.drop(columns=['isFraud', 'TransactionID', 'TransactionDT', 'week'])

# all three should have 431 columns (434 + week = 435, minus 4)
print(X_train.shape, X_val.shape, X_test.shape)

(431125, 431) (77090, 431) (82325, 431)


**Result:** `X_train` (431,125 x 431), `X_val` (77,090 x 431), `X_test` (82,325 x 431). All three have the same 431 columns (435 - 4: `df_merged` has 434 columns plus the `week` column I added), so the model sees the same clues everywhere.

## 4. Drop almost-empty columns

**My rule:** drop columns that are more than 90% missing. Not lower, because the device columns are about 76% missing and still useful: transactions with device data had 7.96% fraud against 2.10% without.

**Decided on train only.** In the EDA I found 12 such columns, but that was on the whole dataset, which includes the test weeks. Using that list would let the test period influence a training decision, so I calculate it again on `X_train`.

In [159]:
# missing % of every column, calculated on X_train only
missing_pct_train = X_train.isnull().mean() * 100

# names of the columns that are more than 90% missing (.index holds the column names)
cols_to_drop = missing_pct_train[missing_pct_train > 90].index

print(len(cols_to_drop))
print(missing_pct_train[cols_to_drop].sort_values(ascending=False))

12
id_24    99.145723
id_25    99.073587
id_08    99.070107
id_07    99.070107
id_26    99.069411
id_21    99.068948
id_22    99.068020
id_23    99.068020
id_27    99.068020
D7       93.565439
dist2    93.099681
id_18    91.889127
dtype: float64


**Result:** 12 columns are more than 90% missing in train:

| Column | Missing in train |
|---|---|
| id_24 | 99.2% |
| id_25, id_07, id_08, id_21, id_22, id_23, id_26, id_27 | 99.1% |
| D7 | 93.6% |
| dist2 | 93.1% |
| id_18 | 91.9% |

They are the same 12 columns as in the EDA, so here the decision would have been the same either way. It still matters to do it on train only, because on other data the two lists can differ.

The same `cols_to_drop` list will be removed from `X_train`, `X_val` and `X_test`, so all three keep exactly the same columns (431 - 12 = 419).

### Remove the 12 columns from all three X tables

The list was decided on train only, but it has to be removed from train, validation and test. If one table kept a column the others do not have, the model would get different clues at test time and break.

Run this cell only once: a second run would try to drop columns that are already gone and give a KeyError.

In [160]:
# drop the same 12 almost-empty columns from all three X tables
X_train = X_train.drop(columns=cols_to_drop)
X_val = X_val.drop(columns=cols_to_drop)
X_test = X_test.drop(columns=cols_to_drop)

# check: each should now have 419 columns (431 - 12)
print(X_train.shape, X_val.shape, X_test.shape)

(431125, 419) (77090, 419) (82325, 419)


**Result:** `X_train` (431,125 x 419), `X_val` (77,090 x 419), `X_test` (82,325 x 419). Same rows as before, and 12 fewer columns in each.

**Next:** sort the 419 columns into numbers and categories (text), because the pipeline treats them differently.

## 5. Numbers and categories

The pipeline treats the two kinds of columns differently: number columns get their gaps filled (and scaled for Logistic Regression), and text columns get **encoded** into numbers. So it needs two lists of column names.

`.dtypes` gives the type of every column. `select_dtypes(include='number')` keeps the number columns, and `exclude='number'` keeps everything else. In pandas 3, text columns can show up as `str` or `object`, so "not a number" is the safe way to catch all of them.

In [161]:
# how many columns of each type
print(X_train.dtypes.value_counts())

# two lists of column names for the pipeline
num_cols = X_train.select_dtypes(include='number').columns
cat_cols = X_train.select_dtypes(exclude='number').columns

print(len(num_cols), len(cat_cols))
print(list(cat_cols))

float64    389
str         29
int64        1
Name: count, dtype: int64
390 29
['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo']


**Result:** 390 number columns (389 float, 1 int) and 29 text columns, 419 in total.

The text columns are: `ProductCD`, `card4`, `card6`, `P_emaildomain`, `R_emaildomain`, `M1`-`M9`, `DeviceType`, `DeviceInfo` and 13 identity columns (`id_12`, `id_15`, `id_16`, `id_28`-`id_31`, `id_33`-`id_38`).

**Something to keep in mind:** Kaggle lists `card1`, `card2`, `card3`, `card5`, `addr1` and `addr2` as categories, but they are stored as numbers (codes like 13926), so they landed in `num_cols`. A code is not a real amount: card 13926 is not "bigger" than card 2755. For now I keep them as numbers, because `card1` alone has thousands of different values and turning each into its own column would explode the table. XGBoost can still split on them. This is a known limitation of the Logistic Regression baseline.

In [162]:
num_cols = X_train.select_dtypes(include='number').columns   # 390 number columns
cat_cols = X_train.select_dtypes(exclude='number').columns   # 29 text columns
print(num_cols)
print(cat_cols)

Index(['TransactionAmt', 'card1', 'card2', 'card3', 'card5', 'addr1', 'addr2',
       'dist1', 'C1', 'C2',
       ...
       'id_06', 'id_09', 'id_10', 'id_11', 'id_13', 'id_14', 'id_17', 'id_19',
       'id_20', 'id_32'],
      dtype='str', length=390)
Index(['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1',
       'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15',
       'id_16', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35',
       'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo'],
      dtype='str')


**Result:** the full lists: 390 number column names and 29 text column names. None of the 12 dropped columns (`dist2`, `id_18`, `id_21`-`id_27`, `D7`, `id_07`, `id_08`) appear, so the drop worked.

**Lesson learned:** the first time I printed these lists it said **400** number columns, and `dist2`, `id_18` etc. were still there. I had re-run the cell that creates `X_train` but not the drop cell after it, so the 12 columns came back. Re-running only this cell didn't help, because it just *reads* `X_train`. **Restart + Run All** fixed it.

Cells run out of order can give wrong results **without any error**, so before trusting a number I always Restart + Run All.

# Day 4: Preprocessing

`X_train` is not ready for a model yet:

1. **Empty cells (NaN).** Logistic Regression cannot handle NaN, it crashes.
2. **29 text columns** (like `ProductCD = "C"`). Models only understand numbers.
3. **Very different scales.** `TransactionAmt` goes up to 31,937 while many columns are just 0 or 1. Logistic Regression is sensitive to this.

**Plan for today:**

| # | Step | Why |
|---|---|---|
| 1 | Fill missing numbers (impute) | the model can't read NaN |
| 2 | Add "was missing" flags | missing device info is itself a fraud clue (7.96% vs 2.10%) |
| 3 | Fill missing text + one-hot encode | turn categories like "C" or "visa" into 0/1 columns |
| 4 | Scale the numbers | put all columns on a similar range |
| 5 | Put it all in one Pipeline | the exact same steps run on train, val and test |
| 6 | Fit on train only | no information from the future leaks in |

## Decision 1: fill numbers with the median, not the mean

Small example: pocket money of 5 friends = 100, 120, 130, 150, 10,000.
- **Mean** = 10,500 / 5 = **2,100**: pulled up by the one huge value, describes nobody.
- **Median** = middle value = **130**: close to what most friends really have.

| Data looks like | Use |
|---|---|
| balanced, no big outliers (e.g. heights) | mean (mean ≈ median anyway) |
| skewed, a few huge values (money, prices) | **median** |

Quick test: if the mean is much bigger than the median, the data is **right-skewed**.

Our `TransactionAmt`: median **$68.77**, mean **$135.03**, max **$31,937**, so it is clearly right-skewed. Money-like columns in fraud data are mostly skewed, and I can't check 390 columns by hand, so I use the **median for all number columns**. It is the safe choice.

## Decision 2: add a "was missing" flag

If I fill an empty device cell with the median, it looks like a normal value and the model can't tell it was empty. But empty device info is a fraud clue (7.96% fraud when present vs 2.10% when absent). So before filling, I add an extra 0/1 column that remembers the cell was missing:

| id_01 (before) | id_01 (after fill) | id_01_was_missing |
|---|---|---|
| -5 | -5 | 0 |
| NaN | -5 (median) | **1** |
| -10 | -10 | 0 |

The model gets both a usable number **and** the clue "this was empty".

## Part A: investigate before fixing

**Question 1:** how many number columns in `X_train` still have at least one empty cell?

How I find it, in 3 steps chained with dots:
1. mark every cell as True (empty) or False (filled)
2. ask each column: do you have **any** True? (one True/False answer per column)
3. add them up: True counts as 1, so the sum = number of columns with gaps

In [163]:
print(X_train[num_cols].isnull().any().sum())

331


**Result Q1: 331 out of 390** number columns have at least one empty cell. Only 59 are completely full (like `TransactionAmt` and `card1`).

Almost every row has a gap somewhere, so deleting rows with gaps would throw away nearly all the data. That's why I **fill** the gaps instead.

How the line works:

| Piece | Does |
|---|---|
| `X_train[num_cols]` | only the number columns (`num_cols` is my variable, so no quotes) |
| `.isnull()` | True/False table: is this cell empty? |
| `.any()` | one answer per column: does it have any gap? |
| `.sum()` | True counts as 1, so this counts the columns with gaps |

Without `.any()`, `.isnull().sum()` gives the number of empty cells **per column** instead.

In [164]:
print(X_train[cat_cols].isnull().any().sum())

28


**Result Q2: 28 out of 29** text columns have gaps, so the text columns need a filling step too. A median makes no sense for text ("the median of visa and mastercard"?), so text gets a different fill (Decision 3 below).

**Which text column is the full one?** `.isnull().sum()` without `.any()` gives the empty count per column. Sorting puts the smallest counts at the top.

In [165]:
# empty cells per text column, smallest first
print(X_train[cat_cols].isnull().sum().sort_values().head(5))

ProductCD             0
card6               826
card4               829
P_emaildomain     66980
M6               132193
dtype: int64


**Result:** `ProductCD` has 0 gaps. Every payment has a product type. `card4` and `card6` are almost full (about 830 gaps out of 431,125).

### Q3 and Q4: how many different values does each text column have?

**Why I check this:** one-hot encoding (explained below) makes **one new column per category**. A column with 5 categories becomes 5 columns. A column with 1,500 categories would become 1,500 columns, which is huge and mostly useless.

`.nunique()` counts the different values in each column (NaN not counted).

In [166]:
# number of different values in each text column, biggest first
print(X_train[cat_cols].nunique().sort_values(ascending=False))

DeviceInfo       1564
id_33             192
id_31             110
id_30              71
R_emaildomain      60
P_emaildomain      59
ProductCD           5
card6               4
id_34               4
card4               4
M4                  3
id_15               3
M1                  2
M8                  2
M7                  2
M6                  2
M5                  2
M3                  2
M2                  2
id_16               2
M9                  2
id_28               2
id_29               2
id_12               2
id_35               2
id_36               2
id_37               2
id_38               2
DeviceType          2
dtype: int64


**Result:**

- `ProductCD` has 5 (C, H, R, S, W), `card4` has 4 (visa, mastercard, ...), `P_emaildomain` has 59
- most other text columns have only 2-4 values (`M1`-`M9` are T/F, `DeviceType` is mobile/desktop)
- **`DeviceInfo` has the most by far: 1,564** different phone/browser models, then `id_33` (screen size, 192) and `id_31` (browser version, 110)

Most of those 1,564 device models appear only a handful of times: only 37 of them appear 100 times or more. A column for a model seen 3 times teaches the model nothing. So rare categories get grouped into one shared **"infrequent"** column (Decision 4).

### Q5: are the numbers on the same scale?

In [167]:
# min, max, mean etc. of two number columns
print(X_train[['TransactionAmt', 'C1']].describe())

       TransactionAmt             C1
count   431125.000000  431125.000000
mean       134.649489      15.722380
std        237.954981     152.368956
min          0.251000       0.000000
25%         43.321000       1.000000
50%         69.000000       1.000000
75%        125.000000       3.000000
max      31937.391000    4685.000000


**Result:** `TransactionAmt` goes from 0.25 to 31,937 (median 69). `C1` goes from 0 to 4,685 (median 1). Other columns are just 0/1. Completely different ranges.

**Why that matters:** Logistic Regression multiplies every column by a weight and adds them up. A column with huge numbers can drown the others just because of its units, not because it is more useful. It also makes the training slow to settle. So I **scale** every number column (Decision 6).

## Part B: decisions

### Decision 3: fill empty text with the word "missing"
For text I can't use a median, so I fill the gap with the word `"missing"`. That way "missing" simply becomes its own category, and the model can still learn that empty device info is a fraud clue.

### Decision 4: one-hot encoding, rare and unknown categories
Models only understand numbers. One-hot turns one text column into several 0/1 columns:

| card4 | card4_visa | card4_mastercard | card4_discover |
|---|---|---|---|
| visa | 1 | 0 | 0 |
| mastercard | 0 | 1 | 0 |

Why not just number them visa=1, mastercard=2? Because then the model would think mastercard is "twice" visa, which means nothing.

- **Rare categories:** categories seen fewer than 100 times in train are grouped into one `infrequent` column (`min_frequency=100`).
- **Unknown categories:** the test weeks contain **140 device models** that never appeared in the train weeks (new phones come out). The encoder must not crash on those. With `handle_unknown='infrequent_if_exist'` they go into the same `infrequent` column. That's what happens in real life: a new phone model shows up and the system still has to score the payment.

### Decision 5: fit on train only
"Fit" means *learn the numbers*: the medians, the category list, the mean and spread for scaling. These are learned from **X_train only**, then the same numbers are used on val and test.

If I learned the median from all the data, information from the test weeks (the "future") would leak into training. The test score would look better than it really is. A real bank can't learn from next month's payments either.

### Decision 6: scale with StandardScaler
For every number column: (value - mean) / standard deviation. After this every column is centred around 0 with a similar spread, so no column wins just because of its units.

## Part C: build the pipeline

### Free some memory first
The processed table will have about 1,050 columns for 431,125 rows. The raw tables from the start of the notebook (`df`, `df1`, `df_merged`, `train`, `val`, `test`) are no longer needed, because everything I need is now in `X_train` / `X_val` / `X_test` and `y_train` / `y_val` / `y_test`. `del` removes them from memory.

(If I need them again: Restart + Run All recreates them.)

In [168]:
import gc

# remove the big tables that are no longer needed
del df, df1, df_merged, train, val, test
gc.collect()   # tell Python to give the freed memory back now

246

### Store the numbers as float32
By default every number uses 8 bytes (float64). float32 uses 4 bytes and still keeps about 7 digits, more than enough for amounts and counts. This halves the memory.

In [169]:
# 4 bytes per number instead of 8: same values, half the memory
X_train[num_cols] = X_train[num_cols].astype('float32')
X_val[num_cols] = X_val[num_cols].astype('float32')
X_test[num_cols] = X_test[num_cols].astype('float32')

### The tools

| Tool | Job |
|---|---|
| `SimpleImputer` | fills empty cells (median for numbers, a fixed word for text) |
| `StandardScaler` | scales number columns |
| `OneHotEncoder` | turns text categories into 0/1 columns |
| `Pipeline` | runs steps **in order**, like an assembly line |
| `ColumnTransformer` | sends the number columns to one pipeline and the text columns to another, then glues the results side by side |

In [170]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

### Number pipeline: fill with median + "was missing" flags, then scale

`add_indicator=True` is Decision 2: for every column that had gaps in train, the imputer adds a 0/1 column that says "this cell was empty". It does this **before** filling, so the clue is kept.

In [171]:
num_pipeline = Pipeline([
    # fill gaps with the column median and add a was-missing 0/1 flag for each column with gaps
    ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
    # (value - mean) / std, so every column has a similar range
    ('scaler', StandardScaler()),
])

### Text pipeline: fill with "missing", then one-hot

In [172]:
cat_pipeline = Pipeline([
    # empty text becomes the word 'missing' (its own category)
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    # one 0/1 column per category; rare (<100 in train) and never-seen categories go to one 'infrequent' column
    ('onehot', OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=100,
                             sparse_output=False, dtype='float32')),
])

### Put the two together
Each entry is (name, which pipeline, which columns).

In [173]:
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols),
])

### Fit on train, then transform all three

- `fit_transform(X_train)`: **learn** the medians, categories, means and spreads from train, and apply them to train
- `transform(X_val)` / `transform(X_test)`: **only apply** what was learned from train. Nothing is learned from val or test.

This takes a minute or two.

In [174]:
# learn from train and apply to train
X_train_prep = preprocessor.fit_transform(X_train)

# only apply (no learning) to val and test
X_val_prep = preprocessor.transform(X_val)
X_test_prep = preprocessor.transform(X_test)

### Check the result
Three things must be true:
1. all three have the **same number of columns** (the model expects the same columns every time)
2. the row counts did not change (431,125 / 77,090 / 82,325)
3. **0 empty cells** left

In [175]:
import numpy as np

print(X_train_prep.shape, X_val_prep.shape, X_test_prep.shape)

# count NaN in each processed table: should all be 0
print(np.isnan(X_train_prep).sum(), np.isnan(X_val_prep).sum(), np.isnan(X_test_prep).sum())

(431125, 1052) (77090, 1052) (82325, 1052)
0 0 0


### What are the new columns called?
`get_feature_names_out()` gives the name of every output column. `num__` columns come from the number pipeline (including the `missingindicator_` flags) and `cat__` columns come from one-hot.

In [176]:
feature_names = preprocessor.get_feature_names_out()
print(len(feature_names))
print(feature_names[:5])     # first few: scaled number columns
print(feature_names[-5:])    # last few: one-hot columns

1052
['num__TransactionAmt' 'num__card1' 'num__card2' 'num__card3' 'num__card5']
['cat__DeviceInfo_rv:52.0' 'cat__DeviceInfo_rv:56.0'
 'cat__DeviceInfo_rv:57.0' 'cat__DeviceInfo_rv:58.0'
 'cat__DeviceInfo_infrequent_sklearn']


## Day 4 summary

| Before | After |
|---|---|
| 419 columns (390 numbers + 29 text) | one all-number table (390 scaled numbers + was-missing flags + one-hot columns) |
| 331 number columns and 28 text columns with gaps | 0 empty cells |
| very different scales (0.25 to 31,937) | every number column centred around 0 |
| text like "visa", "C" | 0/1 columns |

Everything was **learned from train only** and applied the same way to val and test.

**Next (Day 5):** the first model, the Logistic Regression baseline, trained on `X_train_prep` with the class imbalance (3.5% fraud) handled through class weights.

# Day 5: Logistic Regression baseline

## Theory first

### What Logistic Regression does
It gives every column a **weight** (how much that clue pushes towards fraud or away from it), adds up *weight x value* for all ~1,050 columns, and squashes the total into a number between 0 and 1 with the sigmoid curve. That number is the **fraud probability**.

Training = finding the weights that make the probabilities match the real `isFraud` answers in train as closely as possible.

### Why a baseline
A baseline is the simplest reasonable model. It gives a number that every fancier model (XGBoost on Day 6) **has to beat**. If XGBoost can't beat it, the extra complexity isn't worth it. Logistic Regression is fast, and its weights are easy to read.

### Why accuracy is useless here
A lazy model that says "not fraud" for every payment is right 96.5% of the time, but catches **zero** fraud.

Example: 100 payments, 4 of them fraud, and the model says "genuine" for all 100.
- accuracy = 96 / 100 = **96%**
- recall = 0 / 4 = **0%**

### The metrics I use instead

| Metric | Question it answers |
|---|---|
| **Recall** | out of all real frauds, how many did I catch? |
| **Precision** | out of all my alerts, how many were really fraud? |
| **PR-AUC** | one number for the precision/recall trade-off over every cut-off. A random guess scores about the fraud rate (~0.03), a perfect model scores 1 |
| **ROC-AUC** | how often a random fraud gets a higher score than a random genuine payment (0.5 = coin flip, 1 = perfect) |
| **Fraud money caught** | out of all the fraud **dollars**, what % did I stop? This is the business number |

### Class imbalance: class weights
Only 3.5% of the training rows are fraud, so the model could get almost everything "right" by ignoring fraud. `class_weight='balanced'` makes every fraud row count about **28 times** more than a genuine row (96.5 / 3.5), so missing a fraud hurts the model as much as all the genuine mistakes together.

In [177]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score, confusion_matrix, precision_score, recall_score

## Train the model

- `class_weight='balanced'`: fraud rows count more (see above)
- `solver='newton-cholesky'`: the method used to find the weights. On a sample it finished in about 15 steps, while the default (`lbfgs`) was still not finished after 1,000 steps. It works well when there are many more rows (431k) than columns (~1,050)
- `max_iter=200`: upper limit of steps, just in case

**Fit on `X_train_prep` / `y_train` only.** This takes a few minutes.

In [178]:
model = LogisticRegression(class_weight='balanced', solver='newton-cholesky', max_iter=200)

# learn the weights from the training weeks only
model.fit(X_train_prep, y_train)

# how many steps it needed (less than 200 = it finished properly)
print(model.n_iter_)

c:\dev\fraudshield-ai\.venv\Lib\site-packages\sklearn\linear_model\_glm\_newton_solver.py:597: LinAlgWarning: The inner solver of NewtonCholeskySolver stumbled upon a singular or very ill-conditioned Hessian matrix at iteration 1. It will now resort to lbfgs instead.
Further options are to use another solver or to avoid such situation in the first place. Possible remedies are removing collinear features of X or increasing the penalization strengths.
The original Linear Algebra message was:
An ill-conditioned matrix detected: slice 0 has rcond = 1.2006838856493118e-09.
  warnings.warn(


[199]


c:\dev\fraudshield-ai\.venv\Lib\site-packages\sklearn\linear_model\_glm\_newton_solver.py:199: ConvergenceWarning: lbfgs failed to converge after 199 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=199).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.iteration += _check_optimize_result("lbfgs", opt_res, max_iter=max_iter)


## Predict on validation

`predict_proba` gives two numbers per row: P(genuine) and P(fraud). `[:, 1]` keeps the second one, the **fraud probability**.

Test stays locked: it is used only once, at the very end of the project.

In [179]:
# fraud probability for every validation payment
val_proba = model.predict_proba(X_val_prep)[:, 1]

print(val_proba[:10])

[0.613049   0.32954586 0.5757987  0.37906677 0.21043491 0.09026203
 0.23080805 0.2742037  0.16770567 0.22425614]


## PR-AUC and ROC-AUC

These use the probabilities directly, so they don't depend on any cut-off. To judge PR-AUC I compare it with the fraud rate in validation (what a random guess would score).

In [180]:
print('PR-AUC :', average_precision_score(y_val, val_proba))
print('ROC-AUC:', roc_auc_score(y_val, val_proba))
print('random guess PR-AUC (fraud rate in val):', y_val.mean())

PR-AUC : 0.37849277418463984
ROC-AUC: 0.8422264975056122
random guess PR-AUC (fraud rate in val): 0.034310546114930604


## Confusion matrix at the 0.5 cut-off

To make a yes/no decision I need a cut-off: probability >= 0.5 means "flag as fraud". (0.5 is just the default. Choosing the cut-off by cost comes later.)

The confusion matrix is laid out like this:

| | predicted genuine | predicted fraud |
|---|---|---|
| **really genuine** | correct | **false alarm** |
| **really fraud** | **missed fraud** | caught fraud |

In [181]:
# 1 = flag as fraud, 0 = let it through
val_pred = (val_proba >= 0.5).astype(int)

print(confusion_matrix(y_val, val_pred))
print('precision:', precision_score(y_val, val_pred))
print('recall   :', recall_score(y_val, val_pred))

[[62952 11493]
 [  811  1834]]
precision: 0.13761536729946724
recall   : 0.6933837429111531


## Fraud money caught

Catching ten $5 frauds matters less than catching one $5,000 fraud. So I also measure the **dollars**:

money caught % = amount of the frauds I flagged / amount of all frauds x 100

`X_val['TransactionAmt']` still has the real dollar amounts (the scaling only happened inside `X_val_prep`).

In [182]:
amount = X_val['TransactionAmt']

# dollars of the frauds the model flagged
caught_money = amount[(y_val == 1) & (val_pred == 1)].sum()

# dollars of all frauds in validation
total_fraud_money = amount[y_val == 1].sum()

print(caught_money, total_fraud_money)
print('fraud money caught %:', caught_money / total_fraud_money * 100)

283711.88 403181.06
fraud money caught %: 70.368355


## Baseline results (validation)

| Metric | Value |
|---|---|
| PR-AUC | **0.378** (random guess: 0.034, so about 11x better) |
| ROC-AUC | **0.842** |
| Precision @ 0.5 | **13.8%**: only about 1 in 7 alerts is really fraud |
| Recall @ 0.5 | **69.3%**: caught 1,834 of the 2,645 frauds |
| Fraud money caught @ 0.5 | **70.4%**: $283,712 of $403,181 |

**Confusion matrix @ 0.5:** 62,952 genuine let through, **11,493 false alarms**, **811 missed frauds**, 1,834 caught frauds.

**How to read it:**
- The ranking is clearly better than random (PR-AUC 0.378 vs 0.034).
- With class weights the model flags a lot: 13,327 alerts to catch 1,834 frauds. An analyst would drown in false alarms at 0.5, which is why the cut-off gets chosen by cost later.

**Honest note about training:** on the full data the solver printed two warnings. Many of the "was missing" flag columns are exact copies of each other (groups of `V` columns are always empty together), so the fast solver switched to `lbfgs`, which then stopped at the 200-step limit before fully finishing. The numbers match the sample test, so they are a fair baseline, but this model is slightly under-trained. Possible fix later: remove duplicate flag columns, or add a bit more regularisation.

**Weak spots of this baseline:**
- it can only learn straight-line effects ("more of X = more fraud"), not combinations like "product C **and** an outlook.com email **and** a new device"
- `card1`, `addr1` etc. are treated as amounts even though they are codes

**Baseline to beat:** PR-AUC **0.378** on the validation weeks.

# Day 6: XGBoost (the challenger)

**Goal:** beat the Logistic Regression baseline on the **same validation weeks** with the **same metrics**.

## Theory first

### Decision tree
A chain of yes/no questions that ends in an answer:

```
ProductCD = C ?
├── yes → amount > 500 ?
│         ├── yes → high fraud risk
│         └── no  → medium risk
└── no  → low risk
```

### Boosting
One small tree is weak. Boosting builds **hundreds of small trees one after another**, and each new tree focuses on the payments the previous trees got **wrong**. The final score adds up all the trees.

### Why it can beat Logistic Regression
- LogReg can only learn straight-line effects: "more of X = more fraud"
- trees learn **combinations**: "product C **and** outlook.com **and** no device info"
- trees learn **jumps**: "risky only between $500 and $2,000"

### What XGBoost does NOT need
- **No scaling.** A tree only asks "is the amount bigger than 500?". Dividing every amount by 100 changes the question to "bigger than 5?" but the same rows go left and right, so the answer doesn't change.
- **No median filling.** XGBoost handles NaN by itself: for every question it learns which side the empty cells should go to. So the "empty = clue" signal is kept automatically.
- **No one-hot.** It can use text columns directly as **categories**.

So XGBoost uses the data from **before** the Day 4 pipeline (`X_train`, `X_val`), not `X_train_prep`.

### Imbalance: `scale_pos_weight`
The same idea as class weights: every fraud row counts *genuine / fraud* times more (about 28 here).

### Overfitting and early stopping
More trees keep improving the **train** score, but after some point they start memorizing train noise and the **validation** score stops improving. Early stopping watches validation after every tree and stops when it hasn't improved for 50 trees in a row.

Honest note: because validation decides when to stop, the validation score is slightly optimistic. That's exactly why test stays locked until the very end.

## Install (once, in the terminal)

```
uv add xgboost
```
Then restart the kernel only if the import below fails.

In [183]:
from xgboost import XGBClassifier

## Text columns as categories

XGBoost needs text columns to have the pandas `category` type. The category list is learned from **train only** (same rule as Day 4). A value in validation that never appeared in train (like a new phone model) becomes NaN, which XGBoost already knows how to handle.

I make copies so `X_train` / `X_val` stay unchanged.

In [184]:
X_train_xgb = X_train.copy()
X_val_xgb = X_val.copy()

for col in cat_cols:
    # learn the category list from train
    X_train_xgb[col] = X_train_xgb[col].astype('category')
    train_categories = X_train_xgb[col].cat.categories

    # validation: values never seen in train become NaN, then use the same category list
    known = X_val_xgb[col].where(X_val_xgb[col].isin(train_categories))
    X_val_xgb[col] = pd.Categorical(known, categories=train_categories)

print(X_train_xgb.dtypes.value_counts())

float32     390
category     12
category      3
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
Name: count, dtype: int64


## scale_pos_weight = genuine rows / fraud rows (train only)

In [185]:
# True counts as 1, so the sums count genuine and fraud rows
pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(pos_weight)

27.457095709570957


## Train with early stopping

| Setting | Meaning |
|---|---|
| `n_estimators=1000` | at most 1,000 trees |
| `learning_rate=0.1` | how much each new tree is allowed to correct (small = careful but needs more trees) |
| `max_depth=6` | each tree asks at most 6 questions in a row |
| `subsample=0.8` | each tree sees a random 80% of the rows (less memorizing) |
| `colsample_bytree=0.8` | each tree sees a random 80% of the columns |
| `tree_method='hist'` | groups numbers into bins first, much faster |
| `enable_categorical=True` | use the category columns directly |
| `eval_metric='aucpr'` | judge progress by PR-AUC |
| `early_stopping_rounds=50` | stop if validation PR-AUC hasn't improved for 50 trees |

`verbose=50` prints the validation PR-AUC every 50 trees, so I can watch it improve.

**This is slow on my laptop: about 20-30 minutes.** Let it run.

In [186]:
xgb_model = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.1,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=pos_weight,
    tree_method='hist',
    enable_categorical=True,
    eval_metric='aucpr',
    early_stopping_rounds=50,
    n_jobs=-1,               # use all CPU cores
)

# learn from train, watch validation for early stopping
xgb_model.fit(X_train_xgb, y_train, eval_set=[(X_val_xgb, y_val)], verbose=50)

# the tree number with the best validation score
print('best tree:', xgb_model.best_iteration)

[0]	validation_0-aucpr:0.29345
[50]	validation_0-aucpr:0.45759
[100]	validation_0-aucpr:0.48371
[150]	validation_0-aucpr:0.50184
[200]	validation_0-aucpr:0.50656
[250]	validation_0-aucpr:0.51070
[300]	validation_0-aucpr:0.51784
[350]	validation_0-aucpr:0.52117
[400]	validation_0-aucpr:0.52040
[402]	validation_0-aucpr:0.52007
best tree: 352


## Evaluate exactly like Day 5

In [187]:
xgb_val_proba = xgb_model.predict_proba(X_val_xgb)[:, 1]
xgb_val_pred = (xgb_val_proba >= 0.5).astype(int)

print('PR-AUC :', average_precision_score(y_val, xgb_val_proba))
print('ROC-AUC:', roc_auc_score(y_val, xgb_val_proba))
print(confusion_matrix(y_val, xgb_val_pred))
print('precision:', precision_score(y_val, xgb_val_pred))
print('recall   :', recall_score(y_val, xgb_val_pred))

xgb_caught_money = amount[(y_val == 1) & (xgb_val_pred == 1)].sum()
print('fraud money caught %:', xgb_caught_money / total_fraud_money * 100)

PR-AUC : 0.5216092359495773
ROC-AUC: 0.8982198248132589
[[70746  3699]
 [  950  1695]]
precision: 0.314238042269188
recall   : 0.6408317580340265
fraud money caught %: 64.058044


## Side by side: Logistic Regression vs XGBoost

`val_proba` / `val_pred` are the Logistic Regression results from Day 5.

In [188]:
comparison = pd.DataFrame({
    'LogReg': [
        average_precision_score(y_val, val_proba),
        roc_auc_score(y_val, val_proba),
        precision_score(y_val, val_pred),
        recall_score(y_val, val_pred),
        caught_money / total_fraud_money * 100,
    ],
    'XGBoost': [
        average_precision_score(y_val, xgb_val_proba),
        roc_auc_score(y_val, xgb_val_proba),
        precision_score(y_val, xgb_val_pred),
        recall_score(y_val, xgb_val_pred),
        xgb_caught_money / total_fraud_money * 100,
    ],
}, index=['PR-AUC', 'ROC-AUC', 'precision @0.5', 'recall @0.5', 'money caught % @0.5'])

print(comparison.round(3))

                     LogReg  XGBoost
PR-AUC                0.378    0.522
ROC-AUC               0.842    0.898
precision @0.5        0.138    0.314
recall @0.5           0.693    0.641
money caught % @0.5  70.368   64.058


**How to read the table (don't skip this):**

- **PR-AUC and ROC-AUC are the fair comparison.** They judge the whole ranking, over every possible cut-off.
- **The @0.5 rows are NOT a fair comparison.** The two models' probabilities live on different scales. In a test on a sample, XGBoost had much higher precision but *lower* recall and money caught at 0.5 than LogReg, even though its PR-AUC was far higher. That doesn't mean it's worse: it just means 0.5 is the wrong cut-off for it.
- That's why the next step is choosing the cut-off **by cost** (missed fraud costs its amount, every alert costs review time) for each model, and only then comparing money caught.

## Which clues does XGBoost use most?

`feature_importances_` = how much each column helped the trees, in total. The bigger, the more the model relies on it.

In [189]:
importance = pd.Series(xgb_model.feature_importances_, index=X_train_xgb.columns)
print(importance.sort_values(ascending=False).head(15))

V258             0.152671
V70              0.076046
V294             0.051887
V201             0.039499
V187             0.033041
V91              0.026420
V30              0.019458
C8               0.016707
V308             0.016519
C14              0.013236
card6            0.011166
R_emaildomain    0.006873
V317             0.006552
V95              0.006469
ProductCD        0.005904
dtype: float32


**How to read it:**
- The top clues are mostly `V` columns (`V258` alone is about 15% of the total importance). These are features Vesta built itself (counts, matches, rankings). Their meaning is hidden, so I can say *that* they matter, not *why*.
- `C8` and `C14` are counts (for example how many addresses are linked to the card).
- Some of my EDA clues show up too: `card6` (credit vs debit), `R_emaildomain` and `ProductCD` are in the top 15. Device columns are not, so the model probably gets that signal through the `V` columns instead.

## XGBoost results (validation)

Training stopped early: the best validation PR-AUC was at **tree 352**. No improvement for the next 50 trees, so it stopped at 402 instead of building all 1,000.

| | LogReg | XGBoost |
|---|---|---|
| PR-AUC | 0.378 | **0.522** |
| ROC-AUC | 0.842 | **0.898** |
| Precision @ 0.5 | 13.8% | **31.4%** |
| Recall @ 0.5 | **69.3%** | 64.1% |
| Money caught @ 0.5 | **70.4%** | 64.1% |
| Alerts @ 0.5 | 13,327 | **5,394** |
| False alarms @ 0.5 | 11,493 | **3,699** |

**Did XGBoost beat the baseline?** Yes, clearly. PR-AUC went from 0.378 to 0.522 (about 38% better), so it puts real frauds much higher up the list.

At 0.5 it catches almost as many frauds (1,695 vs 1,834) with about **3x fewer false alarms**. LogReg still catches a bit more money at 0.5, but only because it flags 2.5 times as many payments. The 0.5 cut-off is not a fair comparison, which is the next step.

**Next:** choose the cut-off by cost for each model, then fraud money caught and precision@K (how many of the top 100 alerts are really fraud).